# Crime Category and Offense Composition

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

import pandas as pd
from crime_snapshot import load_crime_snapshot
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

df, metadata = load_crime_snapshot(
    PROJECT_ROOT / "data" / "processed" / "crime"
)

display(df.head())

In [ ]:
required_columns = [
    "offense_id",
    "offense_date",
    "report_number",
    "offense_category",
    "offense_sub_category",
    "nibrs_crime_against_category",
    "nibrs_group_a_b",
    "nibrs_offense_code_description",
    "nibrs_offense_code",
    "shooting_type_group",
    "neighborhood",
    "precinct",
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

missing_columns

In [ ]:
classification_columns = [
    "offense_category",
    "offense_sub_category",
    "nibrs_crime_against_category",
    "nibrs_group_a_b",
    "nibrs_offense_code_description",
    "nibrs_offense_code",
    "shooting_type_group",
]

category_missingness = (
    df[classification_columns]
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
)

category_missingness

In [ ]:
offense_category_summary = (
    df
    .groupby("offense_category", dropna=False)
    .agg(
        offense_count=("offense_id", "size"),
        unique_reports=("report_number", "nunique"),
    )
    .reset_index()
    .sort_values("offense_count", ascending=False)
)

offense_category_summary["offense_share_percent"] = (
    offense_category_summary["offense_count"]
    / offense_category_summary["offense_count"].sum()
    * 100
).round(2)

offense_category_summary

fig = px.bar(
    offense_category_summary.head(20),
    x="offense_category",
    y="offense_count",
    title="Top Crime Categories",
    template="plotly_dark",
)

fig.update_layout(
    xaxis_title="Offense Category",
    yaxis_title="Reported Offenses",
    xaxis_tickangle=-45,
)

fig.show()

In [ ]:
offense_sub_category_summary = (
    df
    .groupby("offense_sub_category", dropna=False)
    .agg(
        offense_count=("offense_id", "size"),
        unique_reports=("report_number", "nunique"),
    )
    .reset_index()
    .sort_values("offense_count", ascending=False)
)

offense_sub_category_summary["offense_share_percent"] = (
    offense_sub_category_summary["offense_count"]
    / offense_sub_category_summary["offense_count"].sum()
    * 100
).round(2)

offense_sub_category_summary.head(30)

fig = px.bar(
    offense_sub_category_summary.head(30),
    x="offense_sub_category",
    y="offense_count",
    title="Top Crime Sub-Categories",
    template="plotly_dark",
)

fig.update_layout(
    xaxis_title="Offense Sub-Category",
    yaxis_title="Reported Offenses",
    xaxis_tickangle=-45,
)

fig.show()

In [ ]:
category_to_subcategory = (
    df
    .groupby(["offense_category", "offense_sub_category"], dropna=False)
    .agg(
        offense_count=("offense_id", "size"),
    )
    .reset_index()
    .sort_values(
        ["offense_category", "offense_count"],
        ascending=[True, False],
    )
)

category_to_subcategory

top_subcategories_by_category = (
    category_to_subcategory
    .groupby("offense_category", group_keys=False)
)

top_subcategories_by_category

In [ ]:
category_vs_nibrs = (
    df
    .groupby(
        ["offense_category", "nibrs_crime_against_category"],
        dropna=False,
    )
    .agg(
        offense_count=("offense_id", "size"),
    )
    .reset_index()
    .sort_values("offense_count", ascending=False)
)

category_vs_nibrs

category_vs_nibrs_pivot = (
    category_vs_nibrs
    .pivot_table(
        index="offense_category",
        columns="nibrs_crime_against_category",
        values="offense_count",
        fill_value=0,
    )
)

category_vs_nibrs_pivot

In [ ]:
category_share = (
    df["offense_category"]
    .value_counts(normalize=True, dropna=False)
    .mul(100)
    .round(2)
)

rare_category_share = category_share[category_share < 1]

rare_category_share

In [ ]:
neighborhood_category_counts = (
    df
    .dropna(subset=["neighborhood"])
    .groupby(["neighborhood", "offense_category"], dropna=False)
    .agg(
        offense_count=("offense_id", "size"),
    )
    .reset_index()
)

neighborhood_totals = (
    neighborhood_category_counts
    .groupby("neighborhood")["offense_count"]
    .sum()
    .reset_index(name="neighborhood_total")
)

neighborhood_category_counts = neighborhood_category_counts.merge(
    neighborhood_totals,
    on="neighborhood",
    how="left",
)

neighborhood_category_counts["category_share_percent"] = (
    neighborhood_category_counts["offense_count"]
    / neighborhood_category_counts["neighborhood_total"]
    * 100
).round(2)

neighborhood_category_counts.sort_values(
    ["neighborhood_total", "offense_count"],
    ascending=[False, False],
)

In [ ]:
shooting_summary = (
    df
    .groupby("shooting_type_group", dropna=False)
    .agg(
        offense_count=("offense_id", "size"),
        unique_reports=("report_number", "nunique"),
    )
    .reset_index()
    .sort_values("offense_count", ascending=False)
)

shooting_summary

In [ ]:
top_subcategories_by_category = (
    category_to_subcategory.groupby("offense_category")["offense_sub_category"]
    .value_counts()
    .reset_index(name="count")
)

top_subcategories_by_category[top_subcategories_by_category['offense_category'] == 'all other']
